In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (peptidereactor)

This notebook curates the **peptidereactor** dataset, in which peptide sequences and their class labels are provided in separate files. The two sources are aligned by row order, merged into a single standardized table, quality-controlled for duplicate consistency, and exported together with dataset metadata.

- **Toxic effect / endpoint:** toxic
- **Source:** peptidereactor
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads peptide sequences** from a FASTA file (`seqs.fasta`).
- **Loads class labels** from a plain text file (`classes.txt`).
- **Aligns sequences and labels** by their original ordering and merges them into a single table.
- **Standardizes the schema** to:
  - `sequence`
  - `label`
- **Checks duplicated sequences**:
  - identical sequences with consistent labels are collapsed,
  - conflicting label assignments are reported as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends dataset-level QC statistics.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`,
  - `metadata.json`.

In [2]:
name_source = "peptidereactor"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_sequence = read_fasta_doc(f"{PATH_INPUT}/{name_source}/hem_hemopi/seqs.fasta")

In [4]:
df_label = pd.read_csv(f"{PATH_INPUT}/{name_source}/hem_hemopi/classes.txt", names=["label"])

- Concatenating dataset

In [5]:
df_peptidereactor = (
    pd.concat([df_sequence, df_label], axis=1)
    [["sequence", "label"]]
)
df_peptidereactor.shape

(1104, 2)

- Checking duplicates

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_peptidereactor, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_full.shape

(1104, 2)

In [8]:
df_errors.shape

(0, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_peptidereactor)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'MIT',
 'year of publication': 2021,
 'last update date': datetime.datetime(2021, 5, 25, 0, 0),
 'download date': Timestamp('2025-06-01 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Experimentally validated, "Characteristic threshold (IC50, MIC, etc.)"',
 'repository or server': 'https://github.com/spaenigs/peptidereactor',
 'publication': 'https://academic.oup.com/nargab/article/3/2/lqab039/6281452',
 'number_of_raw_sequences': 1104,
 'number_of_sequences_retained': 1104,
 'number_of_positive_sequences': 522,
 'number_of_negative_sequences': 582,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)